# 1. 从本地工具到 MCP Server

### 🌟 MCP Server 能力建设路线
> 这条路线本质上是 MCP Server 能力建设的第三步：

1. **官方远程服务**（聚合数据-天气预报，`11th-weather-mcp`）✅
2. **自定义 Server 并运行**（Dify 知识库查询，`13th-knowledge-base-mcp`）✅
3. **运行开源代码提供服务**（本期目标）🚀

---

### 🎯 本期目标
- **现状**：上一期（`13th`）中，Wikipedia 查询仍以本地 `wikipedia_tool.py` 的 LangChain 工具方式接入。
- **改进**：切换为调用开源 `wikipedia-mcp` 服务，让 Agent 通过 **MCP 协议**访问 Wikipedia。

### 💡 迁移收益
- **工具能力解耦**：查询能力从业务代码中彻底抽离。
- **复用性更强**：多个 Agent（或多个项目）可以共用同一个 Wikipedia MCP 服务。
- **组合更灵活**：可与天气、知识库等其他 MCP 服务轻松并联。

# 2. 这期视频需要掌握的核心知识点

### 🧑‍💻 MCP 的角色分工
- **MCP Server**：提供工具能力（即“能做什么”）。
- **MCP Client**：发现并调用工具（即“桥梁”）。
- **Agent**：负责决策何时调用工具（即“大脑”）。

### 🔍 `wikipedia-mcp` 提供的能力
- **核心功能**：搜索词条、获取摘要、拉取全文、提取章节、发现相关主题。
- **语言支持**：支持多语言和地区映射。
- **传输模式**：支持 `stdio`、`http`、`streamable-http` 多种传输模式。

### 🔗 LangChain 接入 MCP 的方式
- 使用 `MultiServerMCPClient` 统一挂载多个 MCP 服务。
- 通过 `client.get_tools()` 将服务转换成 Agent 可直接使用的工具列表。

### 🔄 改造思路（从“自定义工具”到“MCP 工具”）
1. 保留 Agent 主流程不变。
2. 替换底层的工具加载逻辑。
3. 保留流式输出与工具调用时的可观测性。

# 3. 下载并安装 wikipedia-mcp（conda 环境）

> **提示**：建议在独立的 Conda 虚拟环境中进行安装，避免依赖冲突。

```bash
cd ~/github-workspace/ai-web/agent-video
conda create -n agent-video python=3.12 -y
conda activate agent-video
git clone https://github.com/rudra-ravi/wikipedia-mcp.git
cd wikipedia-mcp
pip install -e .
```

# 4. 启动 wikipedia-mcp 服务（建议用于演示）

> **注意**：建议开启一个单独的终端用于启动服务，方便演示时查看日志。

```bash
conda activate agent-video
wikipedia-mcp --transport streamable-http --host 127.0.0.1 --port 8080 --path /mcp --language zh
```

### ⚙️ 常用可选参数
- `--country Taiwan` / `--country Japan`：按地区自动映射语言。
- `--enable-cache`：开启缓存，减少重复查询开销，提升响应速度。
- `--access-token <token>`：配置 Token 可降低高频调用被限流的风险。

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

async def load_mcp_tools():
    client = MultiServerMCPClient(
        {
            "wikipedia": {
                "transport": "streamable-http",
                "url": "http://127.0.0.1:8080/mcp"
            },
            "juhe_mcp": {
                "transport": "sse",
                "url": "https://mcp.juhe.cn/sse?token=LOtUYoDAXDf5bGZjTGESDO5w6NmCOXZcLnwqfxXvZcZVV6"
            }
        }
    )
    return await client.get_tools()

# 5. 迁移改造示例

### 🛠️ 动手实践步骤
1. **复制并重命名**：复制 `13th-knowledge-base-mcp` 目录下的代码，重新命名为 `14th-wikipedia-mcp`。
2. **启动服务**：将 `wikipedia-mcp` 项目以一个单独终端运行并提供服务。
3. **修改代码**：参考下方示例，替换工具加载逻辑。

### 📝 代码示例
```python
import asyncio
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

def _load_mcp_tools():
    async def _load():
        client = MultiServerMCPClient(
            {
                "wikipedia": {
                    "transport": "streamable-http",
                    "url": "http://127.0.0.1:8080/mcp"
                }
            }
        )
        return await client.get_tools()
    return asyncio.run(_load())

def build_agent(openai_api_key: str, base_url: str, system_prompt: str):
    mcp_tools = _load_mcp_tools()
    return create_agent(
        model=ChatOpenAI(model="gpt-5-mini", api_key=openai_api_key, base_url=base_url),
        tools=[*mcp_tools],
        system_prompt=system_prompt,
    )
```